In [47]:
%pip install openpyxl

import pandas as pd
import re
import boto3

# Lê o Excel original
file_path = 'Synthetic Booking Emails Modifications/synthetic_booking_emails_original.xlsx'
df = pd.read_excel(file_path)
print("Original:")
print(df.head())

df.to_csv('synthetic_booking_emails_original.csv', index=False, encoding='utf-8')

# Corrige encoding
def corrigir_encoding(cell):
    try:
        # Só corrige se claramente estiver corrompido
        return cell.encode('latin1').decode('utf-8')
    except Exception:
        return cell

# Aplicar a função para corrigir encoding
df_corrigido = df.map(lambda cell: corrigir_encoding(cell) if isinstance(cell, str) else cell)

print("Depois da possível correção de encoding:")
print(df.head())

# Normalizar todos os hífens estranhos e casos corrompidos provavelmente numa conversao feita anteriormente ao dataset nos ter sido disponibilizado
HIFENS_UNICODE = r'[\u2010\u2011\u2012\u2013\u2014\u2015\u2212]'
def normalizar_hifens(texto):
    if not isinstance(texto, str):
        return texto
    texto = re.sub(HIFENS_UNICODE, '-', texto)  # hífens unicode
    texto = texto.replace('â€‘', '-')           # casos já corrompidos
    return texto

# Aplica normalização a todo o df
df_corrigido = df_corrigido.map(normalizar_hifens)

#df_corrigido = df_corrigido.drop(columns=[col for col in df_corrigido.columns if "Unnamed" in col])

print("Após a normalização de hífens:")
print(df_corrigido.head())

Note: you may need to restart the kernel to use updated packages.
Original:
            email_id    platform                         subject  \
0  rentalcars_126225  Rentalcars  A reserva foi criada - 126225.   
1  rentalcars_198246  Rentalcars  A reserva foi criada - 198246.   
2  rentalcars_948749  Rentalcars  A reserva foi criada - 948749.   
3  rentalcars_197251  Rentalcars  A reserva foi criada - 197251.   
4  rentalcars_182627  Rentalcars  A reserva foi criada - 182627.   

                 to                                               body  
0  info@renticop.pt  Caro(a) InÃªs Santos,\n\nObrigado por reservar...  
1  info@renticop.pt  Caro(a) John Silva,\n\nObrigado por reservar c...  
2  info@renticop.pt  Caro(a) Tiago Smith,\n\nObrigado por reservar ...  
3  info@renticop.pt  Caro(a) Miguel Santos,\n\nObrigado por reserva...  
4  info@renticop.pt  Caro(a) Joana Marques,\n\nObrigado por reserva...  
Depois da possível correção de encoding:
            email_id    platform    

In [48]:
# Guarda como CSV e JSON
df_corrigido.to_excel("synthetic_booking_emails.xlsx", index=False)
df_corrigido.to_json("synthetic_booking_emails.json", orient='records', indent=2, force_ascii=False)

# Enviar ficheiros para o S3
s3_client = boto3.client('s3', region_name='eu-west-1')
bucket_name = 'i32419'

def upload_file(local_file_path, s3_path):
    s3_client.upload_file(local_file_path, bucket_name, s3_path)
    print(f"Arquivo {local_file_path} enviado para s3://{bucket_name}/{s3_path}")

upload_file('synthetic_booking_emails.xlsx', 'datasets/synthetic_booking_emails.xlsx')
upload_file('synthetic_booking_emails.json', 'datasets/synthetic_booking_emails.json')

Arquivo synthetic_booking_emails.xlsx enviado para s3://i32419/datasets/synthetic_booking_emails.xlsx
Arquivo synthetic_booking_emails.json enviado para s3://i32419/datasets/synthetic_booking_emails.json
